# 딥러닝응용III — Lecture 03 실습 Part 2
# WordPiece & BPE vs WordPiece 비교 (교수용 — 답안 포함)

| 항목 | 내용 |
|------|------|
| **강의** | 딥러닝응용III (자연어처리) |
| **수업** | 3주차 — 토큰화 기초 Part 2 |
| **교수** | 유원상 교수 |

> **교수용** — 모든 문제의 답안과 풀이가 포함되어 있습니다.

### 🎯 학습 목표
1. **WordPiece 어휘 집합 구축** — `BertWordPieceTokenizer`, `vocab.txt`
2. **`##` prefix 원리** — 어절 내 서브워드 위치 표시
3. **BERT 입력값** — `input_ids`, `attention_mask`, `token_type_ids`
4. **BPE vs WordPiece** — 빈도 vs 우도(likelihood) 비교

---
## 셀 1. 실습 환경 만들기

런타임 → 하드웨어 가속기 → **None (CPU)**

### ❓ Q1-1. 패키지 역할

> (a) `BertWordPieceTokenizer`가 포함된 패키지는? (korpora / tokenizers / transformers)
>
> (b) `BertTokenizer`가 포함된 패키지는?

**🔑 답:** (a) **tokenizers** (어휘 집합 학습) (b) **transformers** (학습된 어휘 로드 및 입력값 생성)

### 📝 Q1-2. 빈칸 채우기 — 패키지 설치

```python
!pip install -q ______ ______ ______   # (a)(b)(c) 세 패키지
```

**🔑 답:** (a) korpora (b) tokenizers (c) transformers

In [ ]:
!pip install -q korpora tokenizers transformers  # 🔑 정답

import tokenizers, transformers
print(f'tokenizers: {tokenizers.__version__}')
print(f'transformers: {transformers.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.2 MB/s eta 0:00:00
tokenizers: 0.22.2
transformers: 5.0.0


---
## 셀 2. 구글 드라이브 연동

Part 1의 BBPE 결과와 Part 2의 WordPiece 결과를 모두 관리합니다.

### 📝 Q1-3. 빈칸 채우기 — 경로 설정

```python
BBPE_DIR      = "/gdrive/My Drive/nlpbook/______"   # (a)
WORDPIECE_DIR = "/gdrive/My Drive/nlpbook/______"   # (b)
os.______(WORDPIECE_DIR, ______=True)                 # (c)(d)
```

**🔑 답:** (a) bbpe (b) wordpiece (c) makedirs (d) exist_ok

In [ ]:
from google.colab import drive
import os
drive.mount('/gdrive', force_remount=True)
BBPE_DIR      = "/gdrive/My Drive/nlpbook/bbpe"          # 🔑 (a) bbpe
WORDPIECE_DIR = "/gdrive/My Drive/nlpbook/wordpiece"     # 🔑 (b) wordpiece
os.makedirs(WORDPIECE_DIR, exist_ok=True)                # 🔑 (c) makedirs (d) exist_ok
print(f"✅ BBPE: {BBPE_DIR}")
print(f"✅ WP:   {WORDPIECE_DIR}")

Mounted at /gdrive
✅ BBPE: /gdrive/My Drive/nlpbook/bbpe
✅ WP:   /gdrive/My Drive/nlpbook/wordpiece


### 🐛 Q1-4. 오류 수정 — 경로

> 아래 코드에는 **2개의 오류**가 있습니다.
>
> ```python
> WP_DIR = "/gdrive/nlpbook/wordpiece"    # 오류 1
> os.makedirs(WP_DIR)                     # 오류 2
> ```

**🔑 답:**
1. `"/gdrive/nlpbook/"` → `"/gdrive/My Drive/nlpbook/"` — `My Drive` 누락
2. `os.makedirs(WP_DIR)` → `os.makedirs(WP_DIR, exist_ok=True)` — 이미 존재 시 에러 방지

### ❓ Q1-5. 저장 위치

> (a) 학습 데이터(train.txt)를 `/root/`에 저장하는 이유는?
>
> (b) 토크나이저 결과물은 구글 드라이브에 저장하는 이유는?

**🔑 답:** (a) /root/는 **읽기 속도가 빠르고** 대용량 반복 읽기에 유리
(b) 학습 결과물은 **런타임 종료 후에도 보존** 필요

---
## 셀 3. 말뭉치 준비

BPE와 WordPiece를 **동일한 조건**에서 비교하기 위해 같은 NSMC를 사용합니다.

In [ ]:
TRAIN_PATH = "/root/train.txt"
TEST_PATH  = "/root/test.txt"

def write_lines(path, lines):
    with open(path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(f'{line}\n')

if not os.path.exists(TRAIN_PATH) or not os.path.exists(TEST_PATH):
    print("말뭉치 파일이 없습니다. NSMC를 다운로드합니다...")
    from Korpora import Korpora
    nsmc = Korpora.load("nsmc", force_download=False)
    write_lines(TRAIN_PATH, nsmc.train.get_all_texts())
    write_lines(TEST_PATH,  nsmc.test.get_all_texts())
    print("✅ 다운로드 및 저장 완료")
else:
    print("✅ 기존 말뭉치 파일을 재사용합니다.")

train_mb = os.path.getsize(TRAIN_PATH) / (1024*1024)
test_mb  = os.path.getsize(TEST_PATH)  / (1024*1024)
print(f"   train.txt : {train_mb:.1f} MB  /  test.txt : {test_mb:.1f} MB")


말뭉치 파일이 없습니다. NSMC를 다운로드합니다...

    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 102MB/s]                             
[nsmc] download ratings_test.txt: 4.90MB [00:00, 55.3MB/s]


✅ 다운로드 및 저장 완료
   train.txt : 12.5 MB  /  test.txt : 4.2 MB


---
## 셀 4. WordPiece 어휘 집합 구축

### BPE vs WordPiece 핵심 비교

| 항목 | BPE | WordPiece |
|------|-----|--------|
| 병합 기준 | **빈도** 최대화: `count(AB)` | **우도** 최대화: `count(AB)/(count(A)×count(B))` |
| 시작 단위 | UTF-8 바이트 (256개) | 유니코드 문자 |
| 어절 경계 | `▁` (Ġ 바이트) | `##` prefix |
| 산출물 | `vocab.json` + `merges.txt` | `vocab.txt` (단일 파일) |
| 특수 토큰 | `[PAD]` 만 | `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]` 5종 |
| 한국어 저장 | 특수 유니코드 → 변환 필요 | **그대로 저장** |
| 대표 모델 | GPT-2, GPT-3 | BERT, ELECTRA |

### ❓ Q2-1. BPE vs WordPiece 병합 기준

> (a) BPE가 "이+다" 쌍을 선호하는 이유는?
>
> (b) WordPiece가 "스럽+다" 쌍을 더 선호하는 이유는?
>
> (c) WordPiece 점수 공식의 분모가 큰 쌍은 어떤 특징을 가지나요?

**🔑 답:** (a) BPE는 **빈도**만 보므로 자주 등장하는 흔한 글자쌍 선택
(b) WP는 **우도** 기준 — "스럽"은 거의 항상 "다"와 함께 등장하므로 독립 등장이 드물어 점수 높음
(c) 각각이 **독립적으로 매우 흔한 글자**인 쌍 (예: 이+다, 하+다)

### ❓ Q2-2. 특수 토큰 5종

> BERT의 5종 특수 토큰을 나열하고 각각의 역할을 한 줄씩 설명하세요.
>
> | 토큰 | 인덱스 | 역할 |
> |------|--------|------|
> | ______ | 0 | ______ |
> | ______ | 1 | ______ |
> | ______ | 2 | ______ |
> | ______ | 3 | ______ |
> | ______ | 4 | ______ |

**🔑 답:** [PAD]=0(패딩), [UNK]=1(미등록 토큰), [CLS]=2(문장 시작/분류), [SEP]=3(문장 끝/구분), [MASK]=4(MLM 학습)

### 📝 Q2-3. 빈칸 채우기 — WordPiece 학습 코드

```python
from tokenizers import ______Tokenizer              # (a) 클래스명
wp = ______Tokenizer(lowercase=______)              # (b)(c)
wp.______(files=[...], vocab_size=______,            # (d)(e)
    wordpieces_prefix="______",                      # (f) 서브워드 구분자
    special_tokens=["[PAD]","[UNK]","[CLS]","[SEP]","[MASK]"])
wp.______(WORDPIECE_DIR)                             # (g) 저장 메서드
```

**🔑 답:** (a) BertWordPiece (b) BertWordPiece (c) False (d) train (e) 10000 (f) ## (g) save_model

In [ ]:
# 저장 경로가 없으면 자동 생성
import os
os.makedirs(WORDPIECE_DIR, exist_ok=True)

from tokenizers import BertWordPieceTokenizer        # 🔑 (a) BertWordPiece
wp_tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False) # 🔑 (b)(c) False
print("🔄 학습 중...")
wp_tokenizer.train(files=[TRAIN_PATH,TEST_PATH],    # 🔑 (d) train
    vocab_size=10000,min_frequency=2,                # 🔑 (e) 10000
    wordpieces_prefix="##",                          # 🔑 (f) ##
    special_tokens=["[PAD]","[UNK]","[CLS]","[SEP]","[MASK]"])
wp_tokenizer.save_model(WORDPIECE_DIR)               # 🔑 (g) save_model
print("✅ 완료")
for f in sorted(os.listdir(WORDPIECE_DIR)):
    print(f"   📄 {f}  ({os.path.getsize(os.path.join(WORDPIECE_DIR,f)):,} bytes)")

🔄 학습 중...
✅ 완료
   📄 vocab.txt  (84,792 bytes)


### 🐛 Q2-4. 오류 수정 — WordPiece 학습 (4개)

> ```python
> from tokenizers import ByteLevelBPETokenizer           # 오류 1
> wp = ByteLevelBPETokenizer(lowercase=True)             # 오류 2
> wp.train(files="train.txt",                            # 오류 3
>     special_tokens=["[PAD]"])                           # 오류 4
> ```

**🔑 답:**
1. ByteLevelBPETokenizer → **BertWordPieceTokenizer**
2. lowercase=True → **False** (한국어)
3. files="train.txt" → **files=["train.txt"]** (리스트)
4. ["[PAD]"] → **["[PAD]","[UNK]","[CLS]","[SEP]","[MASK]"]** (5종 필수)

### 🐛 Q2-5. 오류 수정 — 파라미터 (3개)

> ```python
> wp = BertWordPieceTokenizer(strip_accents=True)        # 오류 1
> wp.train(files=[...], wordpieces_prefix="_",           # 오류 2
>     min_frequency=0)                                    # 오류 3
> ```

**🔑 답:**
1. strip_accents=True → **False** (악센트 유지 권장)
2. wordpieces_prefix="_" → **"##"** (BERT 표준)
3. min_frequency=0 → **2** (노이즈 방지)

### ❓ Q2-6. 산출물 비교

> (a) BPE의 산출물 파일 2개는?
>
> (b) WordPiece의 산출물 파일은?
>
> (c) WordPiece에 `merges.txt`가 없는 이유는?
>
> (d) WordPiece에서 한국어가 깨지지 않는 이유는?

**🔑 답:** (a) vocab.json + merges.txt (b) **vocab.txt** (단일 파일) (c) **최장 일치(longest match)** 방식이므로 어휘 집합만 필요 (d) 문자 단위로 시작하여 한국어를 **그대로** 저장 (BPE는 바이트→특수 유니코드)

---
### 🔍 `vocab.txt` 구조

한 줄에 토큰 하나, **줄 번호(0-indexed) = 토큰 인덱스**.
`##`으로 시작하는 토큰은 **어절 내 중간 위치** 서브워드.

In [ ]:
# WordPiece 산출물 확인  ← Explain 핵심 셀 ②
#
# vocab.txt 구조:
#   - 한 줄에 토큰 하나 / 줄 번호(0-indexed) = 토큰 인덱스
#   - 처음 5개: 특수 토큰 [PAD]=0, [UNK]=1, [CLS]=2, [SEP]=3, [MASK]=4
#   - 이후: 단일 문자 → 자주 등장하는 단어 → 서브워드 순
#   - ##으로 시작하는 줄: 어절 내 중간 위치의 서브워드 (어절 첫 토큰이 아님)
with open(f"{WORDPIECE_DIR}/vocab.txt", 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"=== vocab.txt ===")
print(f"총 어휘 수: {len(lines):,}개")

print("\n[ 처음 10개 — 특수 토큰 + 초기 단일 문자 ]")
print(f"  {'인덱스':>6}   토큰")
print("  " + "─" * 30)
for i, line in enumerate(lines[:10]):
    print(f"  {i:6d}   {line.rstrip()}")

print("\n[ ## prefix 토큰 예시 — 어절 내 서브워드 ]")
print(f"  {'인덱스':>6}   토큰")
print("  " + "─" * 30)
count = 0
for i, line in enumerate(lines):
    if line.startswith("##"):
        print(f"  {i:6d}   {line.rstrip()}")
        count += 1
        if count >= 10: break

total_subword = sum(1 for l in lines if l.startswith("##"))
print(f"\n  전체 ## prefix 토큰 수: {total_subword:,}개  "
      f"({total_subword/len(lines)*100:.1f}%)")

print("\n[ 마지막 5개 — 자주 등장하는 긴 서브워드 ]")
for i, line in enumerate(lines[-5:], start=len(lines)-5):
    print(f"  {i:6d}   {line.rstrip()}")


=== vocab.txt ===
총 어휘 수: 10,000개

[ 처음 10개 — 특수 토큰 + 초기 단일 문자 ]
     인덱스   토큰
  ──────────────────────────────
       0   [PAD]
       1   [UNK]
       2   [CLS]
       3   [SEP]
       4   [MASK]
       5   !
       6   "
       7   %
       8   &
       9   '

[ ## prefix 토큰 예시 — 어절 내 서브워드 ]
     인덱스   토큰
  ──────────────────────────────
    1005   ##입
    1006   ##사
    1007   ##를
    1008   ##망
    1009   ##한
    1010   ##거
    1011   ##작
    1012   ##이
    1013   ##구
    1014   ##먼

  전체 ## prefix 토큰 수: 3,532개  (35.3%)

[ 마지막 5개 — 자주 등장하는 긴 서브워드 ]
    9995   에일리언
    9996   99
    9997   very
    9998   ㅠㅠㅠㅠㅠ
    9999   간간히


### ❓ Q2-7. vocab.txt 해석

> (a) `[PAD]`의 인덱스는? `[CLS]`의 인덱스는?
>
> (b) `##`으로 시작하는 토큰의 의미는?
>
> (c) `짜증`과 `##나네요`가 vocab.txt에 있을 때, "짜증나네요"는 어떻게 토큰화?

**🔑 답:** (a) [PAD]=**0**, [CLS]=**2** (줄 번호=인덱스) (b) 어절 내 **첫 서브워드가 아닌** 토큰 (c) `["짜증", "##나네요"]` — 첫 서브워드에 ## 없음

### 📝 Q2-8. 빈칸 — vocab.txt 인덱스

> vocab.txt의 처음 5줄:
> 0번줄=[______], 1번줄=[______], 2번줄=[______], 3번줄=[______], 4번줄=[______]

**🔑 답:** [PAD], [UNK], [CLS], [SEP], [MASK]

In [ ]:
# WordPiece 동작 확인 — ## prefix 원리 이해
#
# WordPiece 토큰화 절차:
#   1. 공백으로 어절 분리
#   2. 각 어절을 vocab.txt 기준으로 최장 일치(longest match)로 서브워드 분리
#   3. 어절의 첫 서브워드: ## 없음 (새 어절 시작)
#      이후 서브워드: ## prefix 부착 (앞 토큰과 이어짐을 표시)
#   4. 어절 전체가 어휘 집합에 없으면: [UNK] 처리

sample_words = [
    "짜증나네요",    # 예상: 짜증나 + ##네요
    "초딩영화줄",    # 예상: 초딩 + ##영화 + ##줄
    "오버연기조차",  # 예상: 오버 + ##연기 + ##조차
    "가볍지",        # 예상: 가볍 + ##지
]

print("=== WordPiece ## prefix 동작 예시 ===")
print("어절 내 위치에 따라 ## prefix 유무가 결정됩니다.")
print()
print(f"  {'입력 어절':15}   {'토큰화 결과'}")
print("  " + "─" * 50)
for word in sample_words:
    result = wp_tokenizer.encode(word)
    print(f"  {word:15}   {result.tokens}")


=== WordPiece ## prefix 동작 예시 ===
어절 내 위치에 따라 ## prefix 유무가 결정됩니다.

  입력 어절             토큰화 결과
  ──────────────────────────────────────────────────
  짜증나네요             ['짜증나', '##네요']
  초딩영화줄             ['초딩', '##영화', '##줄']
  오버연기조차            ['오버', '##연기', '##조차']
  가볍지               ['가볍', '##지']


### 🧮 Q2-9. ## prefix 해석

> 위 출력에서 "짜증나네요" → `["짜증", "##나", "##네요"]`라면:
>
> (a) 어절의 첫 서브워드는?
>
> (b) `##나`에 ##이 붙은 이유는?
>
> (c) 토큰들에서 ##을 제거하고 합치면 원래 어절이 복원되나요?

**🔑 답:** (a) **"짜증"** (## 없음 = 어절 시작) (b) 어절의 첫 토큰이 아니라 앞 토큰에 **이어지는** 서브워드 (c) 네 — "짜증"+"나"+"네요"="짜증나네요" 복원 가능

---
## 셀 5. BPE vs WordPiece 병합 점수 비교

**BPE 점수** = `count(ab)` — 단순 빈도

**WordPiece 점수** = `count(ab) / (count(a) × count(b))` — 우도 기반 (교재 수식 2-1)

### 📝 Q3-1. 빈칸 — 점수 공식

```python
score_bpe = ______                             # (a) BPE 점수
score_wp  = ______ / (______ * ______)         # (b)(c)(d) WP 점수
```

**🔑 답:** (a) cnt_ab (b) cnt_ab (c) cnt_a (d) cnt_b

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 1: 말뭉치에서 문자 단위 유니그램·바이그램 빈도 집계
#
# 토크나이저 학습 첫 단계를 재현합니다.
# 실제 BPE/WordPiece는 어절(공백 분리) 내에서 문자 단위로 시작하여
# 가장 점수가 높은 인접 쌍을 하나씩 병합해 나갑니다.
#
# 여기서는 그 초기 상태(모든 문자가 개별 토큰인 상태)에서
# BPE 점수와 WordPiece 점수를 직접 계산하여 비교합니다.
# ──────────────────────────────────────────────────────────────
from collections import Counter
import math

print("📊 NSMC train.txt에서 문자 빈도를 집계합니다...")

unigram_counter = Counter()   # 단일 문자 빈도: count(a)
bigram_counter  = Counter()   # 인접 문자 쌍 빈도: count(ab)
total_chars     = 0

with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 30000: break   # 속도를 위해 3만 문장만 사용
        for word in line.split():
            chars = list(word)
            # 유니그램 집계
            unigram_counter.update(chars)
            total_chars += len(chars)
            # 바이그램 집계 (어절 경계를 넘지 않음)
            for j in range(len(chars) - 1):
                bigram_counter[(chars[j], chars[j+1])] += 1

n = total_chars

print(f"✅ 집계 완료")
print(f"   총 문자 수(n)      : {n:,}")
print(f"   유니그램 종류     : {len(unigram_counter):,}개")
print(f"   바이그램 종류     : {len(bigram_counter):,}개")
print(f"\n  [ 가장 많이 등장한 유니그램 Top 10 ]")
print(f"  {'문자':>6}   {'빈도':>10}")
print("  " + "─" * 25)
for ch, cnt in unigram_counter.most_common(10):
    print(f"  {ch:>6}   {cnt:>10,}")


# ──────────────────────────────────────────────────────────────
# STEP 2: BPE 점수 & WordPiece 점수 계산
#
# BPE 점수   = count(ab)                       ← 단순 빈도
# WP  점수   = count(ab) / (count(a) × count(b)) ← 우도 기반
# ──────────────────────────────────────────────────────────────

scores = []
for (a, b), cnt_ab in bigram_counter.items():
    cnt_a = unigram_counter[a]
    cnt_b = unigram_counter[b]

    score_bpe = cnt_ab                             # BPE: 단순 빈도
    score_wp  = cnt_ab / (cnt_a * cnt_b)           # WordPiece: 우도 기반

    scores.append({
        "pair"     : f"{a} + {b}",
        "a"        : a,
        "b"        : b,
        "cnt_ab"   : cnt_ab,
        "cnt_a"    : cnt_a,
        "cnt_b"    : cnt_b,
        "score_bpe": score_bpe,
        "score_wp" : score_wp,
    })

# ── BPE 기준 상위 20개 ────────────────────────────────────────
bpe_top = sorted(scores, key=lambda x: x["score_bpe"], reverse=True)[:20]
wp_top  = sorted(scores, key=lambda x: x["score_wp"],  reverse=True)[:20]

print("=" * 70)
print("  BPE 기준 상위 20개 쌍  (단순 빈도: count(ab) 높은 순)")
print("=" * 70)
print(f"  {'쌍':>8}   {'count(ab)':>10}   {'count(a)':>10}   {'count(b)':>10}   {'WP 점수':>12}")
print("  " + "─" * 62)
for r in bpe_top:
    print(f"  {r['pair']:>8}   {r['cnt_ab']:>10,}   {r['cnt_a']:>10,}   {r['cnt_b']:>10,}   {r['score_wp']:>12.6f}")


📊 NSMC train.txt에서 문자 빈도를 집계합니다...
✅ 집계 완료
   총 문자 수(n)      : 864,761
   유니그램 종류     : 2,075개
   바이그램 종류     : 58,348개

  [ 가장 많이 등장한 유니그램 Top 10 ]
      문자           빈도
  ─────────────────────────
       .       48,223
       이       27,806
       다       22,008
       는       18,286
       고       14,414
       지       13,455
       화       13,267
       영       12,990
       가       11,495
       하       11,461
  BPE 기준 상위 20개 쌍  (단순 빈도: count(ab) 높은 순)
         쌍    count(ab)     count(a)     count(b)          WP 점수
  ──────────────────────────────────────────────────────────────
     . + .       21,575       48,223       48,223       0.000009
     영 + 화       11,882       12,990       13,267       0.000069
     다 + .        8,011       22,008       48,223       0.000008
     ㅋ + ㅋ        4,430        6,792        6,792       0.000096
     ! + !        2,806        6,263        6,263       0.000072
     니 + 다        2,605        6,020       22,008       0.000020
     는 + 데        

### ❓ Q3-2. 점수 차이 해석

> (a) BPE Top에 "이+다"가 있지만 WordPiece Top에는 없는 이유?
>
> (b) WordPiece Top에 드문 쌍이 올라오는 이유?
>
> (c) 두 알고리즘이 같은 쌍을 1순위로 선택하나요?

**🔑 답:** (a) "이","다" 각각 매우 흔함 → 분모 커서 WP 점수 낮아짐
(b) 각각 독립 등장이 드문 쌍 → 분모 작아서 WP 점수 높아짐
(c) 대부분 **다른 쌍**을 선택 → 최종 어휘 집합이 달라짐

### 🐛 Q3-3. 오류 — 점수 계산 (2개)

> ```python
> score_bpe = cnt_a                              # 오류 1
> score_wp  = cnt_ab / (cnt_a + cnt_b)           # 오류 2
> ```

**🔑 답:**
1. cnt_a → **cnt_ab** — BPE는 바이그램 쌍의 **빈도** count(ab)
2. cnt_a **+** cnt_b → cnt_a **×** cnt_b — WordPiece 분모는 **곱** (교재 수식 2-1)

### 🧮 Q3-4. WordPiece 점수 직접 계산

> count(ab)=100, count(a)=500, count(b)=200일 때:
>
> (a) BPE 점수 = ______
>
> (b) WP 점수 = 100 / (______ × ______) = ______
>
> (c) count(ab)=50, count(a)=60, count(b)=55일 때 WP 점수 = ______

**🔑 답:** (a) **100** (b) 100/(500×200)=**0.001** (c) 50/(60×55)=50/3300≈**0.0152** — (c)가 빈도는 낮지만 WP 점수는 15배 높음!

### 📝 Q3-5. 빈칸 — 왜 다른 쌍을 선택하나?

> | 상황 | BPE 판단 | WordPiece 판단 |
> |------|---------|---------------|
> | `이+다` (각자 매우 흔함) | ______ | ______ |
> | `스럽+다` (각자 드뭄) | ______ | ______ |

**🔑 답:** 이+다: BPE="자주 붙으니 병합", WP="각자도 흔하니 굳이?"
스럽+다: BPE="별로 안 보이니 패스", WP="거의 항상 붙으니 병합!"

---
## 셀 6. BERT 입력값 만들기

| 입력값 | GPT | BERT |
|--------|-----|------|
| `input_ids` | 토큰 인덱스 | **`[CLS]` 앞 + `[SEP]` 뒤** 자동 추가 |
| `attention_mask` | 실제=1, PAD=0 | 동일 |
| `token_type_ids` | **없음** | 첫 문장=`0`, 두 번째 문장=`1` |

### 📝 Q4-1. 빈칸 — BERT 토크나이저 로드

```python
from transformers import ______                   # (a) 클래스
tok = ______.from_pretrained(WORDPIECE_DIR,       # (b)
      do_lower_case=______)                        # (c) 학습 시 lowercase와 일치
```

**🔑 답:** (a) BertTokenizer (b) BertTokenizer (c) False

In [ ]:
from transformers import BertTokenizer             # 🔑 (a) BertTokenizer
tokenizer_bert = BertTokenizer.from_pretrained(    # 🔑 (b) BertTokenizer
    WORDPIECE_DIR, do_lower_case=False)             # 🔑 (c) False
print(f"✅ 어휘: {len(tokenizer_bert):,}개")
print(f"   [PAD]={tokenizer_bert.pad_token_id} [CLS]={tokenizer_bert.cls_token_id} [SEP]={tokenizer_bert.sep_token_id}")

✅ 어휘: 10,000개
   [PAD]=0 [CLS]=2 [SEP]=3


### 🐛 Q4-2. 오류 — BERT 로드 (3개)

> ```python
> from transformers import GPT2Tokenizer                 # 오류 1
> tok = GPT2Tokenizer.from_pretrained(WORDPIECE_DIR)     # 오류 2
> tok.pad_token = "[PAD]"                                # 오류 3: 불필요?
> ```

**🔑 답:**
1. GPT2Tokenizer → **BertTokenizer** (WordPiece=BertTokenizer)
2. GPT2Tokenizer → **BertTokenizer** + `do_lower_case=False` 추가
3. BertTokenizer는 pad_token이 이미 있으므로 **불필요** (GPT와 다름)

### 🐛 Q4-3. 오류 — 토크나이저 혼용 (2개)

> ```python
> # WordPiece를 GPT 토크나이저로 로드
> GPT2Tokenizer.from_pretrained(WORDPIECE_DIR)    # 오류 1
> # BPE를 BERT 토크나이저로 로드
> BertTokenizer.from_pretrained(BBPE_DIR)          # 오류 2
> ```

**🔑 답:**
1. WordPiece(vocab.txt) → **BertTokenizer** 필요 (GPT2Tokenizer는 vocab.json+merges.txt 기대)
2. BPE(vocab.json+merges.txt) → **GPT2Tokenizer** 필요 (BertTokenizer는 vocab.txt 기대)

### ❓ Q4-4. 특수 토큰과 입력값

> (a) `[CLS]`의 인덱스와 역할은?
>
> (b) `[SEP]`의 인덱스와 역할은?
>
> (c) GPT에는 없지만 BERT에만 있는 입력값은?
>
> (d) `do_lower_case=False`를 쓰는 이유는?

**🔑 답:** (a) 인덱스 **2**, 문서 분류용 벡터 위치
(b) 인덱스 **3**, 문장 끝/구분
(c) **token_type_ids** (문장 구분)
(d) 학습 시 lowercase=False로 학습했으므로 **일치**시켜야 함

In [ ]:
# BERT 토크나이저 — 예시 문장 토큰화  ← Explain 핵심 셀 ③
#
# ## prefix 해석:
#   ## 없는 토큰: 어절의 첫 번째 서브워드 (새 어절 시작)
#   ## 있는 토큰: 앞 토큰에 이어지는 서브워드 (어절 계속)

sentences = [
    "아 더빙.. 진짜 짜증나네요 목소리",
    "흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나",
    "별루 였다..",
]

print("=== BERT 토크나이저 토큰화 결과 ===")
print("## 없음: 새 어절 시작 | [##토큰]: 앞 토큰에 이어지는 서브워드")
for i, sent in enumerate(sentences, 1):
    tokens = tokenizer_bert.tokenize(sent)
    highlighted = [f"[{t}]" if t.startswith("##") else t for t in tokens]
    print(f"\n[문장{i}] {sent}")
    print(f"  토큰 수 : {len(tokens)}")
    print(f"  토큰    : {highlighted}")


=== BERT 토크나이저 토큰화 결과 ===
## 없음: 새 어절 시작 | [##토큰]: 앞 토큰에 이어지는 서브워드

[문장1] 아 더빙.. 진짜 짜증나네요 목소리
  토큰 수 : 8
  토큰    : ['아', '더빙', '.', '.', '진짜', '짜증나', '[##네요]', '목소리']

[문장2] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
  토큰 수 : 20
  토큰    : ['흠', '.', '.', '.', '포스터', '[##보고]', '초딩', '[##영화]', '[##줄]', '.', '.', '.', '.', '오버', '[##연기]', '[##조차]', '가볍', '[##지]', '않', '[##구나]']

[문장3] 별루 였다..
  토큰 수 : 4
  토큰    : ['별루', '였다', '.', '.']


### 🧮 Q4-5. 토큰화 결과 해석

> 위 출력을 보고 답하세요.
>
> (a) `##`이 붙은 토큰의 의미는?
>
> (b) 문장1의 토큰 수는?
>
> (c) "짜증나네요"가 어떻게 분리되었나요?

**🔑 답:** (a) 어절 내부 서브워드 (첫 토큰이 아님)
(b) 출력 확인
(c) 출력 확인 (예: ["짜증나", "##네요"])

### 📝 Q4-6. 빈칸 — BERT 배치 인코딩

```python
batch = tokenizer_bert(______, padding="______",  # (a)(b)
    max_length=______, truncation=______)            # (c)(d)
```

**🔑 답:** (a) sentences (b) max_length (c) 12 (d) True

In [ ]:
batch_bert = tokenizer_bert(sentences, padding="max_length",  # 🔑 (a)(b)
    max_length=12, truncation=True)                              # 🔑 (c)(d)

In [ ]:
# BERT 모델 입력 만들기 — 단일 문장
#
# 내부 동작:
#   tokenize → [CLS] 앞 추가, [SEP] 뒤 추가 → indexing → padding/truncation
#
# max_length=12: 실제 모델에서는 512 사용; 표 표시 편의를 위해 12로 설정

batch_bert = tokenizer_bert(
    sentences,
    padding="max_length",
    max_length=12,
    truncation=True,
)

header = "구분    " + "  ".join([f"토큰{i:02d}" for i in range(1, 13)])
sep    = "─" * 82

print("=== BERT input_ids ===")
print(f"  앞에 [CLS](id={tokenizer_bert.cls_token_id})가 자동 추가됩니다.")
print(f"  뒤에 [SEP](id={tokenizer_bert.sep_token_id})가 자동 추가됩니다.")
print(f"  0 = [PAD] 토큰")
print()
print(header); print(sep)
for i, ids in enumerate(batch_bert['input_ids'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in ids]))

# 문장1 상세 — 인덱스 → 토큰 문자열 역변환
print()
print("[ 문장1 상세: 각 위치의 인덱스와 토큰 ]")
ids1    = batch_bert['input_ids'][0]
tokens1 = tokenizer_bert.convert_ids_to_tokens(ids1)
print(f"  {'위치':>4}   {'id':>6}   토큰")
print("  " + "─" * 35)
for pos, (tid, tok) in enumerate(zip(ids1, tokens1), 1):
    marker = " ← [CLS]" if tok=="[CLS]" else " ← [SEP]" if tok=="[SEP]" else              " ← [PAD]" if tok=="[PAD]" else " ← 서브워드" if tok.startswith("##") else ""
    print(f"  {pos:4d}   {tid:6d}   {tok}{marker}")

print()
print("=== BERT attention_mask ===")
print("  1 = 실제 토큰 ([CLS],[SEP] 포함)  /  0 = 패딩")
print()
print(header); print(sep)
for i, mask in enumerate(batch_bert['attention_mask'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in mask]))

print()
print("=== BERT token_type_ids ===")
print("  단일 문장 입력이므로 모두 0  (두 문장 입력 시 두 번째 문장 위치는 1)")
print()
print(header); print(sep)
for i, ttids in enumerate(batch_bert['token_type_ids'], 1):
    print(f"문장{i}  " + "  ".join([f"{x:5d}" for x in ttids]))


=== BERT input_ids ===
  앞에 [CLS](id=2)가 자동 추가됩니다.
  뒤에 [SEP](id=3)가 자동 추가됩니다.
  0 = [PAD] 토큰

구분    토큰01  토큰02  토큰03  토큰04  토큰05  토큰06  토큰07  토큰08  토큰09  토큰10  토큰11  토큰12
──────────────────────────────────────────────────────────────────────────────────
문장1      2    621   2631     16     16   1993   3678   1990   3323      3      0      0
문장2      2    997     16     16     16   2609   2045   2796   1981   1277     16      3
문장3      2   3274   9508     16     16      3      0      0      0      0      0      0

[ 문장1 상세: 각 위치의 인덱스와 토큰 ]
    위치       id   토큰
  ───────────────────────────────────
     1        2   [CLS] ← [CLS]
     2      621   아
     3     2631   더빙
     4       16   .
     5       16   .
     6     1993   진짜
     7     3678   짜증나
     8     1990   ##네요 ← 서브워드
     9     3323   목소리
    10        3   [SEP] ← [SEP]
    11        0   [PAD] ← [PAD]
    12        0   [PAD] ← [PAD]

=== BERT attention_mask ===
  1 = 실제 토큰 ([CLS],[SEP] 포함)  /  0 = 패딩

구분    토큰01  토큰02  토큰0

### 🧮 Q4-7. input_ids 해석

> (a) 모든 문장의 첫 값이 `2`인 이유는?
>
> (b) `3`이 나타나는 위치의 의미는?
>
> (c) GPT input_ids vs BERT input_ids의 차이 1가지?
>
> (d) `max_length=12`에서 BERT 실제 콘텐츠 토큰 최대 수는? (힌트: [CLS]+[SEP])

**🔑 답:** (a) 2=[CLS] 자동 추가
(b) 3=[SEP] 문장 끝 표시
(c) BERT는 [CLS]+[SEP] 자동 추가
(d) **10개** (12 - [CLS] 1개 - [SEP] 1개)

### 📝 Q4-8. 빈칸 — attention_mask 작성

> `max_length=10`, 실제 토큰 수 6개([CLS]+4개+[SEP])일 때:
>
> attention_mask = [______, ______, ______, ______, ______, ______, ______, ______, ______, ______]

**🔑 답:** [1,1,1,1,1,1,0,0,0,0] — 실제 6개=1, 패딩 4개=0

### ❓ Q4-9. token_type_ids

> (a) 단일 문장 입력 시 token_type_ids 값은 모두 ______
>
> (b) 두 문장 입력 시 문장A 영역의 값은 ______, 문장B 영역의 값은 ______
>
> (c) GPT에 token_type_ids가 없는 이유는?

**🔑 답:** (a) **0** (세그먼트 0)
(b) **0**, **1**
(c) GPT는 단일 문장만 입력받는 자기회귀 모델로, 문장 구분이 불필요

---
### 두 문장 입력 — `token_type_ids` 실제 동작

In [ ]:
# 두 문장 동시 입력 — token_type_ids 확인
#
# tokenizer_bert(문장A, 문장B) 형태로 쌍을 전달하면
# 내부적으로 "[CLS] 문장A [SEP] 문장B [SEP]" 구조로 구성됩니다.
#
# token_type_ids:
#   첫 번째 세그먼트 ([CLS] 포함 ~ 첫 [SEP] 포함) : 0
#   두 번째 세그먼트 (문장B 시작 ~ 두 번째 [SEP] 포함) : 1

sent_a = "이 영화 재미있었어요"
sent_b = "정말 추천합니다"

pair_input = tokenizer_bert(
    sent_a, sent_b,
    padding="max_length",
    max_length=16,
    truncation=True,
)

ids    = pair_input['input_ids']
ttids  = pair_input['token_type_ids']
mask   = pair_input['attention_mask']
tokens = tokenizer_bert.convert_ids_to_tokens(ids)

print(f"=== 두 문장 동시 입력 ===")
print(f"  문장A: '{sent_a}'")
print(f"  문장B: '{sent_b}'")
print(f"  입력 형식: [CLS] 문장A [SEP] 문장B [SEP]")
print()
print(f"  {'위치':>4}   {'id':>6}   {'type':>6}   {'att':>5}   토큰   ← 세그먼트")
print("  " + "─" * 58)
for pos, (tid, tok, ttype, att) in enumerate(zip(ids, tokens, ttids, mask), 1):
    if att == 0:
        seg = "[PAD]"
    elif ttype == 0:
        seg = "세그먼트 0  (문장A)"
    else:
        seg = "세그먼트 1  (문장B)"
    print(f"  {pos:4d}   {tid:6d}   {ttype:6d}   {att:5d}   {tok:12}  ← {seg}")


=== 두 문장 동시 입력 ===
  문장A: '이 영화 재미있었어요'
  문장B: '정말 추천합니다'
  입력 형식: [CLS] 문장A [SEP] 문장B [SEP]

    위치       id     type     att   토큰   ← 세그먼트
  ──────────────────────────────────────────────────────────
     1        2        0       1   [CLS]         ← 세그먼트 0  (문장A)
     2      711        0       1   이             ← 세그먼트 0  (문장A)
     3     1979        0       1   영화            ← 세그먼트 0  (문장A)
     4     4823        0       1   재미있었어요        ← 세그먼트 0  (문장A)
     5        3        0       1   [SEP]         ← 세그먼트 0  (문장A)
     6     1988        1       1   정말            ← 세그먼트 1  (문장B)
     7     3972        1       1   추천합니다         ← 세그먼트 1  (문장B)
     8        3        1       1   [SEP]         ← 세그먼트 1  (문장B)
     9        0        0       0   [PAD]         ← [PAD]
    10        0        0       0   [PAD]         ← [PAD]
    11        0        0       0   [PAD]         ← [PAD]
    12        0        0       0   [PAD]         ← [PAD]
    13        0        0       0   [PAD]         ←

### 🧮 Q4-10. 두 문장 입력 해석

> (a) `[CLS] 문장A [SEP] 문장B [SEP]` — [SEP]이 2개인 이유는?
>
> (b) 첫 번째 [SEP]의 token_type_ids = ______ (0 또는 1?)
>
> (c) 두 번째 [SEP]의 token_type_ids = ______ (0 또는 1?)
>
> (d) `token_type_ids`에서 0→1 전환 경계는 어디?

**🔑 답:** (a) 첫 [SEP]은 문장A의 끝, 두 번째 [SEP]은 문장B의 끝
(b) **0** (문장A 소속)
(c) **1** (문장B 소속)
(d) 첫 [SEP] **직후** (문장B 시작 지점)

### 📝 Q4-11. 빈칸 — BERT 입력 형식

> 단일 문장: `[______] 문장A [______]`
>
> 두 문장: `[______] 문장A [______] 문장B [______]`

**🔑 답:** 단일: [CLS] 문장A [SEP] / 두 문장: [CLS] 문장A [SEP] 문장B [SEP]

### 🐛 Q4-12. 오류 — 두 문장 입력

> ```python
> # 두 문장을 리스트로 넘김
> pair = tokenizer_bert([sent_a, sent_b])  # 오류
> ```
>
> 위 코드의 오류와 올바른 코드를 쓰세요.

**🔑 답:**
`tokenizer_bert([sent_a, sent_b])` → `tokenizer_bert(sent_a, sent_b)` — 두 문장을 **별도 인자**로 전달해야 쌍 입력으로 인식. 리스트로 넘기면 **배치 인코딩**(2개 독립 문장)으로 처리됨.

---
## 셀 7. BPE vs WordPiece 직접 비교

In [ ]:
# BPE vs WordPiece 비교 준비
# ── decode_bbpe_token 재정의 (Part 1 런타임 종료 대비) ────────
def bytes_to_unicode():
    bs = (list(range(ord("!"), ord("~") + 1))
          + list(range(ord("¡"), ord("¬") + 1))
          + list(range(ord("®"), ord("ÿ") + 1)))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}

def decode_bbpe_token(token: str) -> str:
    """BBPE 내부 표현 토큰을 한국어로 변환. 불완전한 UTF-8은 hex로 표시."""
    try:
        byte_list = [byte_decoder[c] for c in token]
    except KeyError:
        return token
    decoded = bytes(byte_list).decode("utf-8", errors="replace")
    if '\ufffd' in decoded:
        return '[' + ' '.join(f'{b:02x}' for b in byte_list) + ']'
    return decoded.replace(' ', '▁')

# ── BPE 토크나이저 로드 ──────────────────────────────────────
from transformers import GPT2Tokenizer

if os.path.exists(BBPE_DIR) and os.listdir(BBPE_DIR):
    tokenizer_gpt = GPT2Tokenizer.from_pretrained(BBPE_DIR)
    tokenizer_gpt.pad_token = "[PAD]"
    print(f"✅ BPE(GPT)  토크나이저 로드 완료  (어휘 집합: {len(tokenizer_gpt):,}개)")
else:
    print("⚠️  BPE 토크나이저가 없습니다. Part 1 노트북을 먼저 실행하세요.")
    tokenizer_gpt = None

print(f"✅ WP(BERT)  토크나이저 확인        (어휘 집합: {len(tokenizer_bert):,}개)")


✅ BPE(GPT)  토크나이저 로드 완료  (어휘 집합: 10,001개)
✅ WP(BERT)  토크나이저 확인        (어휘 집합: 10,000개)


In [ ]:
# BPE vs WordPiece 토큰화 직접 비교
compare_sentences = [
    "아 더빙.. 진짜 짜증나네요 목소리",
    "흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나",
    "별루 였다..",
    "딥러닝을 공부하는 것은 매우 흥미롭습니다",
    "인공지능이 세상을 바꾸고 있습니다",
]

print("=" * 75)
print("              BPE(GPT) vs WordPiece(BERT) 토큰화 비교")
print("=" * 75)

for i, sent in enumerate(compare_sentences, 1):
    print(f"\n[문장{i}] {sent}")
    print("─" * 65)

    if tokenizer_gpt:
        bpe_tokens   = tokenizer_gpt.tokenize(sent)
        bpe_readable = [decode_bbpe_token(t) for t in bpe_tokens]
        print(f"  BPE   ({len(bpe_tokens):2d}토큰) : {bpe_readable}")
    else:
        print("  BPE   : (Part 1 먼저 실행 필요)")

    wp_tokens  = tokenizer_bert.tokenize(sent)
    wp_display = [f"[{t}]" if t.startswith("##") else t for t in wp_tokens]
    print(f"  WP    ({len(wp_tokens):2d}토큰) : {wp_display}")


              BPE(GPT) vs WordPiece(BERT) 토큰화 비교

[문장1] 아 더빙.. 진짜 짜증나네요 목소리
─────────────────────────────────────────────────────────────────
  BPE   ( 7토큰) : ['아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']
  WP    ( 8토큰) : ['아', '더빙', '.', '.', '진짜', '짜증나', '[##네요]', '목소리']

[문장2] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
─────────────────────────────────────────────────────────────────
  BPE   (15토큰) : ['흠', '...', '포스터', '보고', '▁초딩', '영화', '줄', '....', '오버', '연기', '조차', '▁가볍', '지', '▁않', '구나']
  WP    (20토큰) : ['흠', '.', '.', '.', '포스터', '[##보고]', '초딩', '[##영화]', '[##줄]', '.', '.', '.', '.', '오버', '[##연기]', '[##조차]', '가볍', '[##지]', '않', '[##구나]']

[문장3] 별루 였다..
─────────────────────────────────────────────────────────────────
  BPE   ( 4토큰) : ['별루', '[20 ec 98]', '[80 eb 8b a4]', '..']
  WP    ( 4토큰) : ['별루', '였다', '.', '.']

[문장4] 딥러닝을 공부하는 것은 매우 흥미롭습니다
─────────────────────────────────────────────────────────────────
  BPE   (11토큰) : ['[eb 94]', '[a5]', '러', '닝', '을', '▁공부', '하는', '▁것은', 

### ❓ Q5-1. 어절 경계 표시 비교

> (a) BPE에서 새 어절이 시작됨을 나타내는 기호는?
>
> (b) WordPiece에서 어절 내부 토큰임을 나타내는 기호는?
>
> (c) 두 방식이 **정반대**인 이유를 한 문장으로 설명하세요.

**🔑 답:** (a) `▁` (Ġ 바이트) — 어절 시작 표시
(b) `##` — 이전 토큰에 이어짐 표시
(c) BPE는 **"시작"을 표시**, WP는 **"이어짐"을 표시** — 반대 방향 접근

### ❓ Q5-2. 토큰 수 비교

> (a) 같은 문장에서 BPE와 WP의 토큰 수가 다른 이유는?
>
> (b) 일반적으로 어떤 쪽이 토큰 수가 더 적나요?

**🔑 답:** (a) 병합 기준이 다르므로 **다른 서브워드 조합**을 선택
(b) vocab_size가 같으면 비슷하지만, WP가 의미 단위 병합을 선호하여 약간 적은 경향

In [ ]:
# 정량 비교 — 테스트 데이터 샘플 500문장 통계
print("=== 테스트 데이터 샘플 500문장 정량 비교 ===")
print("(잠시 기다려 주세요...)")

bpe_lengths, wp_lengths = [], []

with open(TEST_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 500: break
        line = line.rstrip()
        if not line: continue
        if tokenizer_gpt:
            bpe_lengths.append(len(tokenizer_gpt.tokenize(line)))
        wp_lengths.append(len(tokenizer_bert.tokenize(line)))

print()
print(f"  {'항목':20}   {'BPE(GPT)':>12}   {'WordPiece(BERT)':>15}")
print("  " + "─" * 55)

if bpe_lengths:
    avg_bpe = sum(bpe_lengths)/len(bpe_lengths)
    med_bpe = sorted(bpe_lengths)[len(bpe_lengths)//2]
    avg_wp  = sum(wp_lengths)/len(wp_lengths)
    med_wp  = sorted(wp_lengths)[len(wp_lengths)//2]
    print(f"  {'평균 토큰 수':20}   {avg_bpe:>12.1f}   {avg_wp:>15.1f}")
    print(f"  {'중앙 토큰 수':20}   {med_bpe:>12d}   {med_wp:>15d}")
    print(f"  {'최대 토큰 수':20}   {max(bpe_lengths):>12d}   {max(wp_lengths):>15d}")
else:
    avg_wp = sum(wp_lengths)/len(wp_lengths)
    print(f"  {'평균 토큰 수(WP)':20}   {'─':>12}   {avg_wp:>15.1f}")

print()
print("=== BPE vs WordPiece 알고리즘 최종 비교 ===")
rows = [
    ("병합 기준",      "빈도(count) 최대화",          "우도(likelihood) 최대화"),
    ("시작 단위",      "UTF-8 바이트",                 "유니코드 문자"),
    ("한국어 저장",    "특수 유니코드로 인코딩",        "그대로 저장"),
    ("어절 경계 표시", "▁ (Ġ 바이트)",                "## prefix"),
    ("산출물 파일",    "vocab.json + merges.txt",     "vocab.txt"),
    ("OOV 처리",      "불가능 (바이트 조합으로 표현)",  "[UNK] 처리"),
    ("특수 토큰",      "[PAD]",                       "[PAD][UNK][CLS][SEP][MASK]"),
    ("추가 입력값",    "input_ids, attention_mask",   "+ token_type_ids"),
    ("대표 모델",      "GPT-2, GPT-3, RoBERTa",      "BERT, ELECTRA, KoBERT"),
]
print(f"\n  {'항목':16}  {'BPE':30}  {'WordPiece'}")
print("  " + "─" * 80)
for item, bpe_val, wp_val in rows:
    print(f"  {item:16}  {bpe_val:30}  {wp_val}")


=== 테스트 데이터 샘플 500문장 정량 비교 ===
(잠시 기다려 주세요...)

  항목                         BPE(GPT)   WordPiece(BERT)
  ───────────────────────────────────────────────────────
  평균 토큰 수                        16.2              16.0
  중앙 토큰 수                          12                12
  최대 토큰 수                          75                70

=== BPE vs WordPiece 알고리즘 최종 비교 ===

  항목                BPE                             WordPiece
  ────────────────────────────────────────────────────────────────────────────────
  병합 기준             빈도(count) 최대화                   우도(likelihood) 최대화
  시작 단위             UTF-8 바이트                       유니코드 문자
  한국어 저장            특수 유니코드로 인코딩                    그대로 저장
  어절 경계 표시          ▁ (Ġ 바이트)                       ## prefix
  산출물 파일            vocab.json + merges.txt         vocab.txt
  OOV 처리            불가능 (바이트 조합으로 표현)               [UNK] 처리
  특수 토큰             [PAD]                           [PAD][UNK][CLS][SEP][MASK]
  추가 입력값            input_ids, at

### 🧮 Q5-3. Shape 계산

> 5개 문장, `max_length=16`으로 BERT 배치 인코딩 시:
>
> (a) `input_ids` shape = (______ × ______)
>
> (b) `token_type_ids` shape = (______ × ______)
>
> (c) `attention_mask` shape = (______ × ______)

**🔑 답:** (a) **(5 × 16)** (b) **(5 × 16)** (c) **(5 × 16)** — 모두 동일 shape

---
## 📝 요약

| 항목 | BPE (GPT) | WordPiece (BERT) |
|------|-----------|------------------|
| 병합 기준 | 빈도 최대화 | 우도 최대화 |
| 시작 단위 | UTF-8 바이트 (256) | 유니코드 문자 |
| 어절 경계 | `▁` (새 어절 시작) | `##` (이전에 이어짐) |
| 산출물 | vocab.json + merges.txt | vocab.txt |
| 특수 토큰 | [PAD] | [PAD][UNK][CLS][SEP][MASK] 5종 |
| 입력값 | input_ids + attention_mask | + **token_type_ids** |
| 대표 모델 | GPT-2, GPT-3 | BERT, ELECTRA |

---
## 📋 종합 퀴즈 — 교수용 답안

### Quiz 1. 병합 기준
> BPE 점수 공식: ______
> WordPiece 점수 공식: ______

**🔑 답:** BPE = count(ab) / WP = count(ab)/(count(a)×count(b))

### Quiz 2. 산출물 파일
> BPE: ______ + ______
> WordPiece: ______

**🔑 답:** vocab.json + merges.txt / vocab.txt

### Quiz 3. 특수 토큰 5종
> [______], [______], [______], [______], [______]

**🔑 답:** [PAD], [UNK], [CLS], [SEP], [MASK]

### Quiz 4. ## prefix
> "짜증나네요" → ["짜증", "______", "______"]

**🔑 답:** ["짜증", "##나", "##네요"] (출력에 따라 다름)

### Quiz 5. BERT 입력 형식

**🔑 답:** 단일: [CLS] A [SEP] / 두 문장: [CLS] A [SEP] B [SEP]

### Quiz 6. token_type_ids

**🔑 답:** 단일: 모두 0 / 두 문장: A=0, B=1

### Quiz 7. 오류 찾기

**🔑 답:** BBPE_DIR에는 vocab.json+merges.txt → **GPT2Tokenizer** 필요

### Quiz 8. 빈칸 채우기

**🔑 답:** BertWordPiece / ## / save_model

### Quiz 9. 어절 경계

**🔑 답:** BPE: ▁ / WP: ##

### Quiz 10. 자동 추가

**🔑 답:** [CLS](2) / [SEP](3)

### Quiz 11. Shape

**🔑 답:** (5 × 16)

### Quiz 12. WP 점수

**🔑 답:** BPE=100, WP=100/(500×200)=0.001